# imports

In [ ]:
import os
import zipfile
import urllib.request
import random
import torchvision
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from skimage.feature import local_binary_pattern
from skimage.filters.rank import entropy
from skimage.morphology import disk
import torchvision.transforms as transforms
from dataset import MoCoTextureDataset

In [ ]:
!git pull

# sync github

In [ ]:
repo_name = "SIV-Texture-Anomaly-detection"
git_url = "https://github.com/Marco1743/SIV-Texture-Anomaly-detection.git"
if not os.path.exists(repo_name):
    !git clone {git_url}
%cd {repo_name}
!git pull

c:\Users\marco\Desktop\SIV\SIV-Texture-Anomaly-detection\SIV-Texture-Anomaly-detection


Cloning into 'SIV-Texture-Anomaly-detection'...


Already up to date.


# import dataset

In [ ]:
os.makedirs('data/train/good', exist_ok=True)
os.makedirs('data/test/anomaly', exist_ok=True)

print("Scaricamento Texture reali (DTD Dataset)...")
dtd_train = torchvision.datasets.DTD(root='./data', split='train', download=True)
dtd_test = torchvision.datasets.DTD(root='./data', split='test', download=True)

Download dataset in corso...
Estrazione...


'gdown' is not recognized as an internal or external command,
operable program or batch file.


Dataset pronto!
Cartelle trovate:


'unzip' is not recognized as an internal or external command,
operable program or batch file.
'ls' is not recognized as an internal or external command,
operable program or batch file.


## creazione train e test

In [ ]:
print("creazione training")
count = 0
for img, label in dtd_train:
    img_gray = img.convert('L')
    img_gray.save(f'data/train/good/tex_{count}.png')
    count += 1

print("Generazione anomalie...")
for i in range(50):
    img, _ = dtd_test[i]
    img_arr = np.array(img.convert('L'))
    h, w = img_arr.shape
    
    tipo_anomalia = random.choice([0, 1])
    
    if tipo_anomalia == 0:
        box_h = random.randint(10, 50)
        box_w = random.randint(10, 50)
        y = random.randint(0, h - box_h)
        x = random.randint(0, w - box_w)
        intensita = random.randint(0, 60)
        
        img_arr[y:y+box_h, x:x+box_w] = intensita
        
    else:
        spessore = random.randint(2, 6)
        lunghezza = random.randint(40, 100)
        
        if random.choice([True, False]):
            y = random.randint(0, h - spessore)
            x = random.randint(0, w - lunghezza)
            img_arr[y:y+spessore, x:x+lunghezza] = 255
        else: 
            y = random.randint(0, h - lunghezza)
            x = random.randint(0, w - spessore)
            img_arr[y:y+lunghezza, x:x+spessore] = 255

    Image.fromarray(img_arr).save(f'data/test/anomaly/anom_{i}.png')

print(f"Dataset pronto e processato con augmentations.py! Totale: {count}")

# PIPELINE 1

## Carichiamo l'immagine

In [ ]:
img_path = 'data/test/anomaly/anom_0.png'
img = cv2.imread(img_path, 0)

## Estrazione LBP

In [ ]:
radius = 3
n_points = 8 * radius
lbp_img = local_binary_pattern(img, n_points, radius, method='uniform')

## Entropia Locale sull'LBP

In [ ]:
# Convertiamo l'LBP in un formato digeribile dal filtro di entropia (uint8 0-255)
lbp_uint8 = cv2.normalize(lbp_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
# Calcoliamo l'entropia (il disordine) in una finestra circolare (raggio 15 pixel)
# Il graffio romperà la regolarità delle strisce, creando un picco di entropia!
entropia_locale = entropy(lbp_uint8, disk(15))

## Segmentazione K-Means sulla mappa di Entropia

In [ ]:
X = entropia_locale.reshape((-1, 1))
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X)
anomaly_mask = kmeans.labels_.reshape(img.shape)
# Trucchetto: Assicuriamoci che l'anomalia sia sempre bianca (1) e lo sfondo nero (0)
# Poiché le anomalie sono piccole, il cluster "anomalo" avrà sempre meno pixel dello sfondo
if np.sum(anomaly_mask) > (anomaly_mask.size / 2):
    anomaly_mask = 1 - anomaly_mask

## Visualizzazione

In [ ]:
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.title("Immagine con Anomalia", fontsize=14)
plt.imshow(img, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title("1. Feature LBP (Cruda)", fontsize=14)
plt.imshow(lbp_img, cmap='viridis')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title("2. Entropia Locale (Il Segreto!)", fontsize=14)
plt.imshow(entropia_locale, cmap='inferno')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title("3. Segmentazione K-Means", fontsize=14)
plt.imshow(anomaly_mask, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
print("Inizializzazione del PyTorch Dataset per MoCo...")
train_dataset = MoCoTextureDataset(data_dir='data/train/good', is_train=True)
view_1, view_2 = train_dataset[5]

to_pil = transforms.ToPILImage()
img_1 = to_pil(view_1)
img_2 = to_pil(view_2)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title("Vista 1 (Augmentation Casuale A)", fontsize=14)
plt.imshow(img_1, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Vista 2 (Augmentation Casuale B)", fontsize=14)
plt.imshow(img_2, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Shape del tensore 1: {view_1.shape}")
print(f"Shape del tensore 2: {view_2.shape}")

In [ ]:
# Sincronizziamo GitHub per scaricare la tua versione corretta con il Resize(224, 224)
!git pull

import torch
from torch.utils.data import DataLoader
from dataset import MoCoTextureDataset
from model import MoCo

print("--- FASE 1: Setup dell'Hardware ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

print("\n--- FASE 2: Inizializzazione Dati ---")
train_dataset = MoCoTextureDataset(data_dir='data/train/good', is_train=True)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
print("DataLoader pronto.")

print("\n--- FASE 3: Caricamento Rete Neurale ---")
model = MoCo(dim=128, K=1024).to(device)
print("Architettura MoCo (ResNet18 + Momentum Queue) istanziata.")

print("\n--- FASE 4: Test del Forward Pass (Il Motore gira!) ---")
view_1, view_2 = next(iter(train_loader))

view_1 = view_1.to(device)
view_2 = view_2.to(device)

logits, labels = model(view_1, view_2)

print("\n✅ TEST SUPERATO: Nessun errore di dimensione!")
print(f"Input Shape: {view_1.shape}  --> [Batch Size, Canali, Altezza, Larghezza]")
print(f"Logits Shape: {logits.shape} --> [Batch Size, 1 (Risposta Giusta) + K (Risposte Sbagliate)]")
print(f"Labels Shape: {labels.shape}     --> [Batch Size]")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from dataset import MoCoTextureDataset
from model import MoCo
import matplotlib.pyplot as plt
from tqdm import tqdm  # <-- La magica barra di caricamento!

print("--- FASE 1: Setup ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

BATCH_SIZE = 32
EPOCHS = 15
LR = 0.001

print("\nPreparazione del DataLoader...")
train_dataset = MoCoTextureDataset(data_dir='data/train/good', is_train=True)

# FIX FONDAMENTALE: num_workers=2 dice a Google Colab di usare 2 core in parallelo per calcolare i tuoi filtri custom!
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True # Velocizza il trasferimento dati verso la GPU
)

model = MoCo(dim=128, K=1024).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"\n🚀 INIZIO ADDESTRAMENTO MOCO ({EPOCHS} Epoche) 🚀")
losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    # Avvolgiamo il train_loader con tqdm per vedere a che punto è il batch!
    loop = tqdm(train_loader, desc=f"Epoca [{epoch+1}/{EPOCHS}]")

    for view_1, view_2 in loop:
        view_1, view_2 = view_1.to(device), view_2.to(device)

        optimizer.zero_grad()
        logits, labels = model(view_1, view_2)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        # Aggiorniamo la barra con la loss in tempo reale
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"-> Fine Epoca {epoch+1} | Loss Media: {avg_loss:.4f}\n")

print("\n✅ ADDESTRAMENTO CONCLUSO!")
torch.save(model.encoder_q.state_dict(), "moco_encoder_weights.pth")

plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS+1), losses, marker='o', linestyle='-', color='b')
plt.title("Curva di Apprendimento (Contrastive Loss)", fontsize=14)
plt.xlabel("Epoca", fontsize=12)
plt.ylabel("Loss InfoNCE", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()